# Maximum Independent Set with Neutral Atoms (Pasqal-style)

This notebook is a CUDA-Q adaptation of Pulser's MWIS/QUBO examples:

- [QAA to solve a MWIS problem](https://pulser.readthedocs.io/en/v1.5.5/tutorials/mwis.html)
- [QAOA and QAA to solve a QUBO problem](https://pulser.readthedocs.io/en/v1.4.0/tutorials/qubo.html)

Key behavior in this version:

- **Direct Pasqal flow**: uses `pasqal-cloud` SDK to fetch live device specs and uses those values in the schedule/layout.
- **QRMI flow**: uses `cudaq.set_target("pasqal", machine="qrmi")`; backend selection comes from Slurm `--qpu`.

In [ ]:
import itertools
import json
import os
from pathlib import Path

import numpy as np

import cudaq
from cudaq.dynamics import Schedule
from cudaq.operators import RydbergHamiltonian, ScalarOperator
from pasqal_cloud import SDK


In [ ]:
def read_pasqal_config():
    cfg = {}
    path = Path.home() / ".pasqal" / "config"
    if not path.exists():
        return cfg
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        cfg[k.strip()] = v.strip()
    return cfg

submission_mode = os.environ.get("PASQAL_SUBMISSION_MODE", "local").lower()
config = read_pasqal_config()
sdk = SDK()

if submission_mode == "qrmi":
    cudaq.set_target("pasqal", machine="qrmi")
    selected_machine = os.environ.get("SLURM_JOB_QPU_RESOURCES", "qrmi").split(",")[0]
    hardware_specs = json.loads(next(iter(sdk.get_device_specs_dict().values())))
else:
    # Local simulation keeps default CUDA-Q target and still loads cloud specs for realistic values.
    selected_machine = os.environ.get("PASQAL_MACHINE_TARGET", "EMU_FRESNEL")

    # Direct mode: ensure CUDA-Q gets PASQAL_AUTH_TOKEN and PASQAL_PROJECT_ID.
    if submission_mode == "direct":
        token = os.environ.get("PASQAL_AUTH_TOKEN") or config.get("token")
        project_id = os.environ.get("PASQAL_PROJECT_ID") or config.get("project_id")
        if token is None:
            username = os.environ.get("PASQAL_USERNAME") or config.get("username")
            password = os.environ.get("PASQAL_PASSWORD") or config.get("password")
            if username and password:
                token = SDK(username=username, password=password).user_token()
        if not token:
            raise RuntimeError(
                "Direct mode requires PASQAL_AUTH_TOKEN (or token/username+password in ~/.pasqal/config)."
            )
        if not project_id:
            raise RuntimeError(
                "Direct mode requires PASQAL_PROJECT_ID (or project_id in ~/.pasqal/config)."
            )
        os.environ["PASQAL_AUTH_TOKEN"] = token
        os.environ["PASQAL_PROJECT_ID"] = project_id
        cudaq.set_target("pasqal", machine=selected_machine)

    specs_by_name = {
        name: (json.loads(spec) if isinstance(spec, str) else spec)
        for name, spec in sdk.get_device_specs_dict().items()
    }
    physical_name = selected_machine.replace("EMU_", "")
    if physical_name not in specs_by_name:
        physical_name = "FRESNEL"
    hardware_specs = specs_by_name[physical_name]

print(f"submission mode: {submission_mode}")
print(f"selected machine: {selected_machine}")
print(f"spec source device: {hardware_specs['name']}")
print("has PASQAL_AUTH_TOKEN:", bool(os.environ.get("PASQAL_AUTH_TOKEN")))
print("has PASQAL_PROJECT_ID:", bool(os.environ.get("PASQAL_PROJECT_ID")))


## 1) Define an MIS instance

We use the 4-node graph structure from Pulser's MWIS tutorial and solve the
Maximum Independent Set (MIS) variant (uniform node weights).

In [ ]:
adjacency = np.array([
    [0, 1, 1, 1],
    [1, 0, 0, 0],
    [1, 0, 0, 1],
    [1, 0, 1, 0],
], dtype=int)
weights = np.ones(4, dtype=int)

def is_independent(z: np.ndarray, A: np.ndarray) -> bool:
    for i in range(len(z)):
        for j in range(i + 1, len(z)):
            if A[i, j] == 1 and z[i] == 1 and z[j] == 1:
                return False
    return True

def score(z: np.ndarray, w: np.ndarray) -> int:
    return int(np.dot(z, w))

solutions = []
for bits in itertools.product([0, 1], repeat=4):
    z = np.array(bits, dtype=int)
    if is_independent(z, adjacency):
        solutions.append(("".join(str(b) for b in bits), score(z, weights)))

best_score = max(s for _, s in solutions)
best_bitstrings = [b for b, s in solutions if s == best_score]

print("Classical optimum score:", best_score)
print("Classical optimum bitstrings:", best_bitstrings)

## 2) Build register from live device specs

We use `min_atom_distance` and `max_radial_distance` from the fetched Pasqal
specs to pick a valid 4-atom layout.

In [ ]:
min_atom_distance_um = float(hardware_specs["min_atom_distance"])
max_radial_distance_um = float(hardware_specs["max_radial_distance"])

d_um = max(1.1 * min_atom_distance_um, 6.0)
if d_um >= max_radial_distance_um:
    d_um = 0.5 * max_radial_distance_um

register_um = np.array([
    [0.0, 0.0],
    [d_um, 0.0],
    [0.0, d_um],
    [-d_um, 0.0],
])
register = [(float(x * 1e-6), float(y * 1e-6)) for x, y in register_um]

print(f"min_atom_distance (um): {min_atom_distance_um}")
print(f"max_radial_distance (um): {max_radial_distance_um}")
print(f"chosen spacing d (um): {d_um}")

## 3) Build sweep from live channel specs and run

We read the global Rydberg channel limits (`max_amp`, `max_abs_detuning`,
`max_duration`) from cloud specs and scale the pulse parameters accordingly.

In [ ]:
rydberg_global = [c for c in hardware_specs["channels"] if c["id"] == "rydberg_global"][0]

max_amp = float(rydberg_global["max_amp"]) * 1e6  # rad/s
max_det = float(rydberg_global["max_abs_detuning"]) * 1e6  # rad/s
max_duration = float(rydberg_global["max_duration"]) * 1e-9  # s

time_max = min(3.0e-6, 0.6 * max_duration)
time_ramp = 0.3 * time_max
time_hold = 0.4 * time_max
steps = [0.0, time_ramp, time_ramp + time_hold, time_max]
schedule = Schedule(steps, ["t"])

omega_max = 0.25 * max_amp
delta_start = -0.5 * max_det
delta_end = 0.5 * max_det

def omega_profile(t):
    tr = float(t.real)
    if tr <= 0.0:
        return 0.0
    if tr < time_ramp:
        return omega_max * tr / time_ramp
    if tr < time_ramp + time_hold:
        return omega_max
    if tr < time_max:
        return omega_max * (1.0 - (tr - time_ramp - time_hold) / time_ramp)
    return 0.0

def delta_profile(t):
    tr = min(max(float(t.real), 0.0), time_max)
    return delta_start + (delta_end - delta_start) * tr / time_max

omega = ScalarOperator(lambda t: omega_profile(t))
phi = ScalarOperator.const(0.0)
delta = ScalarOperator(lambda t: delta_profile(t))

shots = int(min(2000, hardware_specs.get("max_runs", 2000)))
result = cudaq.evolve(
    RydbergHamiltonian(
        atom_sites=register,
        amplitude=omega,
        phase=phi,
        delta_global=delta,
    ),
    schedule=schedule,
    shots_count=shots,
)

print("used shots:", shots)
print("used omega_max (rad/s):", omega_max)
print("used detuning range (rad/s):", (delta_start, delta_end))
result.dump()

## 4) Compare to classical optimum

Check whether the highest-frequency bitstrings in `result.dump()` include
the classical optimum list from above: `best_bitstrings`.